# Dataset 3 — Embedding Model Selection (p = 0)

Same selection logic as Dataset 1's `05_embedding_selection_p0_*`, applied to the **static** Dataset 3.
Compares the Node2Vec (network_based) **v1 + v2** candidates at 32 / 64 / 128 dims (GraphSAGE is not applicable to a structure-only network) — and picks the best embedding + model. `ModelTrainer` uses `split='random'`
(same as the combined threshold notebooks). Best model saved to `models/dataset_3/05_a`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    CLASSICAL_FEATURE_CANDIDATES,
    load_gnn_dataset,
    load_model,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
TARGET_COL = 'log_systemic_risk_label'
print('Project root:', PROJECT_ROOT)

Project root: /Users/rubenmarques/Documents/Repositórios/Thesis


## Load embedding candidates (Node2Vec v1 + v2, 32/64/128)

In [2]:
DIMS = [32, 64, 128]
EMB_FILES = {}
for v in ['v1', 'v2']:
    for d in DIMS:
        EMB_FILES[f'node2vec_{v}_{d}'] = f'node2vec_{v}_{d}_dataset3_dataset.parquet'

trainers = {}
for key, fname in EMB_FILES.items():
    edf, ecols = load_gnn_dataset(PROJECT_ROOT, target_col=TARGET_COL, filename=fname)
    trainers[key] = ModelTrainer(df=edf, feature_cols=ecols, target_col=TARGET_COL, split='random')

pd.DataFrame({k: {'n_emb': len(t.feature_cols), 'train': len(t.train_df), 'val': len(t.val_df), 'test': len(t.test_df)}
             for k, t in trainers.items()}).T

,n_emb,train,val,test
node2vec_v1_32,32,7000,1500,1500
node2vec_v1_64,64,7000,1500,1500
node2vec_v1_128,128,7000,1500,1500
node2vec_v2_32,32,7000,1500,1500
node2vec_v2_64,64,7000,1500,1500
node2vec_v2_128,128,7000,1500,1500


## Define Models

In [ ]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(32,63,16), max_iter=300, activation="relu", learning_rate="adaptive", learning_rate_init=0.001, early_stopping=True, n_iter_no_change=3, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}.     crfrfrfr
list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train all candidates

In [4]:
for key, t in trainers.items():
    t.train_all(candidate_models)
    print(f'\n=== {key} ===')
    display(t.leaderboard().assign(**{'val/train_rmse': lambda d: (d['validation_rmse'] / d['train_rmse']).round(2), 'val/train_mae': lambda d: (d['validation_mae'] / d['train_mae']).round(2)})[DISPLAY_COLS + ['val/train_rmse', 'val/train_mae']])


=== node2vec_v1_32 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse,val/train_mae
0,Gradient Boosting,0.245,0.511,0.347,0.725,2.09,2.09
1,MLP,0.506,0.529,0.717,0.739,1.03,1.05
2,XGBoost,0.384,0.541,0.559,0.744,1.33,1.41
3,Random Forest,0.220,0.598,0.294,0.778,2.65,2.72
4,Ridge,0.742,0.745,0.960,0.963,1.00,1.00
5,Linear Regression,0.742,0.745,0.960,0.963,1.00,1.00



=== node2vec_v1_64 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse,val/train_mae
0,Gradient Boosting,0.219,0.533,0.303,0.750,2.48,2.43
1,MLP,0.463,0.532,0.671,0.761,1.13,1.15
2,XGBoost,0.369,0.571,0.533,0.774,1.45,1.55
3,Random Forest,0.240,0.640,0.309,0.822,2.66,2.67
4,Ridge,0.732,0.744,0.946,0.959,1.01,1.02
5,Linear Regression,0.732,0.744,0.946,0.959,1.01,1.02



=== node2vec_v1_128 ===


...

,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse,val/train_mae
0,MLP,0.458,0.498,0.669,0.719,1.07,1.09
1,Gradient Boosting,0.248,0.500,0.351,0.719,2.05,2.02
2,XGBoost,0.379,0.541,0.552,0.745,1.35,1.43
3,Random Forest,0.216,0.582,0.288,0.773,2.68,2.69
4,Ridge,0.747,0.747,0.965,0.967,1.00,1.00
5,Linear Regression,0.746,0.747,0.965,0.967,1.00,1.00



=== node2vec_v2_64 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse,val/train_mae
0,Gradient Boosting,0.215,0.511,0.297,0.720,2.42,2.38
1,MLP,0.457,0.519,0.662,0.746,1.13,1.14
2,XGBoost,0.373,0.571,0.542,0.775,1.43,1.53
3,Random Forest,0.240,0.637,0.307,0.806,2.63,2.65
4,Ridge,0.736,0.741,0.949,0.960,1.01,1.01
5,Linear Regression,0.735,0.741,0.949,0.960,1.01,1.01



=== node2vec_v2_128 ===


,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse,val/train_mae
0,Gradient Boosting,0.187,0.520,0.253,0.715,2.83,2.78
1,MLP,0.404,0.514,0.592,0.748,1.26,1.27
2,XGBoost,0.350,0.569,0.502,0.763,1.52,1.63
3,Random Forest,0.247,0.664,0.313,0.826,2.64,2.69
4,Ridge,0.690,0.711,0.903,0.930,1.03,1.03
5,Linear Regression,0.690,0.711,0.903,0.930,1.03,1.03


## Hyperparameter Tuning

In [5]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([np.full(len(trainer.train_df), -1), np.zeros(len(trainer.val_df), dtype=int)])
    search = RandomizedSearchCV(base_model, param_distributions, n_iter=n_iter,
                                cv=PredefinedSplit(split_idx), scoring='neg_root_mean_squared_error',
                                random_state=42, n_jobs=-1)
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_

RF_PARAMS = {
    'model__n_estimators': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [None, 5, 10, 15, 20, 30],
    'model__min_samples_leaf': [1, 2, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__max_features': ['sqrt', 'log2', 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    'model__max_iter': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [3, 4, 5, 6, 8, None],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__min_samples_leaf': [5, 10, 20, 50, 100],
    'model__l2_regularization': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__max_leaf_nodes': [15, 20, 30, 40, 50, 60],
    'model__max_bins': [64, 128, 255],
}
XGB_PARAMS = {
    'model__n_estimators': [100, 200, 400, 600, 800],
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'model__min_child_weight': [1, 2, 5, 10],
    'model__gamma': [0, 0.1, 0.5, 1.0, 2.0],
    'model__reg_alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__reg_lambda': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 2.0],
}
MLP_PARAMS = {
    'model__hidden_layer_sizes': [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    'model__learning_rate_init': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    'model__learning_rate': ['constant', 'adaptive'],
    'model__batch_size': [32, 64, 128, 'auto'],
}

In [6]:
for key, t in trainers.items():
    print(f'Tuning {key} ...')
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  'Random Forest (tuned)')
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  'Gradient Boosting (tuned)')
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, 'XGBoost (tuned)')
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, 'MLP (tuned)')

Tuning node2vec_v1_32 ...


Tuning node2vec_v1_64 ...
Tuning node2vec_v1_128 ...
Tuning node2vec_v2_32 ...
Tuning node2vec_v2_64 ...
Tuning node2vec_v2_128 ...


## Compare embeddings & pick best

In [7]:
summary = []
for key, t in trainers.items():
    row = t.leaderboard().iloc[0]
    summary.append({'embedding': key, 'model': row['model'],
                    'train_rmse': row['train_rmse'], 'validation_rmse': row['validation_rmse'],
                    'val/train_rmse': round(row['validation_rmse'] / row['train_rmse'], 2),
                    'train_mae': row['train_mae'], 'validation_mae': row['validation_mae'],
                    'val/train_mae': round(row['validation_mae'] / row['train_mae'], 2)})
summary = pd.DataFrame(summary).sort_values('validation_rmse').reset_index(drop=True)
display(summary)
best = summary.iloc[0]
print('Best embedding+model:', best['embedding'], '|', best['model'], '| val_rmse', best['validation_rmse'])

,embedding,model,train_rmse,validation_rmse,val/train_rmse,train_mae,validation_mae,val/train_mae
0,node2vec_v2_32,MLP (tuned),0.577,0.682,1.18,0.380,0.451,1.19
1,node2vec_v2_128,Gradient Boosting (tuned),0.219,0.696,3.18,0.159,0.509,3.20
2,node2vec_v2_64,Gradient Boosting (tuned),0.261,0.703,2.69,0.188,0.501,2.66
3,node2vec_v1_32,MLP (tuned),0.616,0.706,1.15,0.407,0.477,1.17
4,node2vec_v1_128,Gradient Boosting (tuned),0.219,0.722,3.30,0.162,0.526,3.25
5,node2vec_v1_64,MLP (tuned),0.603,0.724,1.20,0.399,0.482,1.21


Best embedding+model: node2vec_v2_32 | MLP (tuned) | val_rmse 0.682


## Save best model

In [8]:
SAVE_DIR = PROJECT_ROOT / 'src' / 'models' / 'dataset_3' / '05_a'
# save the best model for each embedding candidate (per-embedding subfolder)
for key, t in trainers.items():
    t.save_model(t.leaderboard().iloc[0]['model'], SAVE_DIR / key)
print('Overall best:', best['embedding'], '|', best['model'])

Saved 'MLP (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/05_a/node2vec_v1_32/MLP_(tuned).joblib
Saved 'MLP (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/05_a/node2vec_v1_64/MLP_(tuned).joblib
Saved 'Gradient Boosting (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/05_a/node2vec_v1_128/Gradient_Boosting_(tuned).joblib
Saved 'MLP (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/05_a/node2vec_v2_32/MLP_(tuned).joblib
Saved 'Gradient Boosting (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/05_a/node2vec_v2_64/Gradient_Boosting_(tuned).joblib
Saved 'Gradient Boosting (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/05_a/node2vec_v2_128/Gradient_Boosting_(tuned).joblib
Overall best: node2vec_v2_32 | MLP (tuned)
